# EDA

## data peperation part

| section | What we do |
| :--- | :--- |
| [1.1 -- uplode data configuration & Splitting](#section-1-1) | Downloading the data, and splitting it into training and test folders |
| [1.2 -- Fundamentals](#section-1-2) | Understand the shape and meaning of the data |
| [1.3 -- Data Validation](#section-1-3) | Ask systematic questions: does the data meet our expectations? |
| [1.4 -- Data Cleaning](#section-1-4) | produce a clean dataframe ready for analysis. |
| [1.5 -- Feature engineering](#section-1-5) | produce a clean dataframe ready for analysis. |


<a id="section-1-1"></a>

## 1.1 uplode data

In [ ]:
# !git clone https://github.com/netanelr6/nyc-tlc-project-dsma-course.git

Cloning into 'nyc-tlc-project-dsma-course'...
remote: Enumerating objects: 2148, done.
remote: Counting objects: 100% (2148/2148), done.
remote: Compressing objects: 100% (1080/1080), done.
remote: Total 2148 (delta 1148), reused 2059 (delta 1059), pack-reused 0 (from 0)
Receiving objects: 100% (2148/2148), 5.48 MiB | 12.44 MiB/s, done.
Resolving deltas: 100% (1148/1148), done.


In [ ]:
# %cd /content/nyc-tlc-project-dsma-course

/content/nyc-tlc-project-dsma-course


In [2]:

import pandas as pd

import joblib
from pathlib import Path


from src.download_data           import run_download_pipeline
from src.validation              import validate_nyc_taxi_data
from src.cleaning                import clean_parquet
# from src.splitting             import split_train_test, subsample_splits # in our project we split the data in the biginning by date and save in sepret folders, so we don't need to split again in the pipeline
from src.features                import run_feature_pipeline, run_baseline_pipeline, TARGET_COL, FEATURE_CREATION_STEPS


# ── Path configuration ────────────────────────────────

RAW_DATA_PARQUET_TEST     = "data/raw/test/"
RAW_DATA_PARQUET_TRAIN    = "data/raw/train/"

# RAW_CLEAN_PARQUET    = "data/processed/"
TRAIN_CLEANED_PARQUET = "data/processed/DF_test_2024_2025_cleaned.parquet"
TEST_CLEANED_PARQUET  = "data/processed/Df_test_2026_cleaned.parquet"

SCALER_SAVE_PATH_BSSELINE = "data/feature_stores/baseline_scaler.pkl"
SCALER_SAVE_PATH_ENGINEERD = "data/feature_stores/engineered_scaler.pkl"
MODEL_DIR_ENGINEERED = "data/models/engineered_model"

# ── Helpers ──────────────────────────────────────

def _print_header(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


In [3]:
run_download_pipeline()

print("Data download step finished.")

INITIALIZING NYC TLC DATA SYNC PIPELINE
  [EXISTS] yellow_tripdata_2024-01.parquet is already locally available.
           URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
  [EXISTS] yellow_tripdata_2024-02.parquet is already locally available.
           URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet
  [EXISTS] yellow_tripdata_2024-03.parquet is already locally available.
           URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet
  [EXISTS] yellow_tripdata_2024-04.parquet is already locally available.
           URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-04.parquet
  [EXISTS] yellow_tripdata_2024-05.parquet is already locally available.
           URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-05.parquet
  [EXISTS] yellow_tripdata_2024-06.parquet is already locally available.
           URL: https://d37ci6vzurychx.clo

<a id="section-1-2"></a>
## 1.2 Fundamentals

In [4]:
df_row_test  = pd.read_parquet(str(RAW_DATA_PARQUET_TEST) , engine='pyarrow')
df_row_train = pd.read_parquet(str(RAW_DATA_PARQUET_TRAIN), engine='pyarrow')

print("-" * 30 + " RAW DATA OVERVIEW " + "-" * 30)


print("-" * 20 + "      TRAIN DATA   " + "-" * 20)
print(f'Loaded {len(df_row_train):,} rows x {df_row_train.shape[1]} columns')
df_row_train.info()


print("-" * 60)

print("-" * 20 + "      test DATA     " + "-" * 20)
print(f'Loaded {len(df_row_test):,} rows x {df_row_test.shape[1]} columns')
df_row_test.info()

------------------------------ RAW DATA OVERVIEW ------------------------------
--------------------      TRAIN DATA   --------------------
Loaded 41,169,720 rows x 19 columns
<class 'pandas.DataFrame'>
RangeIndex: 41169720 entries, 0 to 41169719
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           floa

In [5]:
df_row_train.head()
# df_row_test.head(5)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [6]:
# Summary statistics for every numeric column
# Look for: surprising min/max values, large std relative to mean, suspicious zeros
df_row_train.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
count,4.116972e+07,41169720,41169720,3.707849e+07,4.116972e+07,3.707849e+07,4.116972e+07,4.116972e+07,4.116972e+07,4.116972e+07,4.116972e+07,4.116972e+07,4.116972e+07,4.116972e+07,4.116972e+07,4.116972e+07,3.707849e+07,3.707849e+07
mean,1.764232e+00,2024-07-06 10:01:25.051412,2024-07-06 10:18:53.125903,1.333931e+00,4.976101e+00,2.322150e+00,1.642428e+02,1.634475e+02,1.107259e+00,1.926851e+01,1.385953e+00,4.797774e-01,3.307884e+00,5.615266e-01,9.629934e-01,2.783281e+01,2.232144e+00,1.470060e-01
min,1.000000e+00,2002-12-31 16:46:07,2002-12-31 17:24:07,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,-2.261200e+03,-9.250000e+00,-5.000000e-01,-3.000000e+02,-1.406300e+02,-1.000000e+00,-2.265450e+03,-2.500000e+00,-1.750000e+00
25%,2.000000e+00,2024-04-06 20:07:27.500000,2024-04-06 20:23:52,1.000000e+00,1.010000e+00,1.000000e+00,1.320000e+02,1.130000e+02,1.000000e+00,9.300000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.575000e+01,2.500000e+00,0.000000e+00
50%,2.000000e+00,2024-07-03 23:35:15,2024-07-03 23:50:35,1.000000e+00,1.760000e+00,1.000000e+00,1.610000e+02,1.620000e+02,1.000000e+00,1.350000e+01,1.000000e+00,5.000000e-01,2.600000e+00,0.000000e+00,1.000000e+00,2.100000e+01,2.500000e+00,0.000000e+00
75%,2.000000e+00,2024-10-08 17:33:35,2024-10-08 17:53:27.250000,1.000000e+00,3.360000e+00,1.000000e+00,2.330000e+02,2.340000e+02,1.000000e+00,2.260000e+01,2.500000e+00,5.000000e-01,4.250000e+00,0.000000e+00,1.000000e+00,3.060000e+01,2.500000e+00,0.000000e+00
max,7.000000e+00,2026-06-26 23:53:12,2026-06-27 20:59:10,9.000000e+00,3.986086e+05,9.900000e+01,2.650000e+02,2.650000e+02,5.000000e+00,3.355444e+05,6.599000e+01,4.130000e+01,9.999900e+02,1.702880e+03,2.000000e+00,3.355509e+05,2.520000e+00,1.750000e+00
std,4.258568e-01,NaN,NaN,8.158242e-01,4.192305e+02,1.092805e+01,6.434069e+01,6.960009e+01,6.515108e-01,7.671984e+01,1.815878e+00,1.301830e-01,4.090523e+00,2.240545e+00,2.550554e-01,7.805359e+01,8.746528e-01,5.020407e-01


## 1.3  Data Validation


In [7]:
# from src.validation import validate_nyc_taxi_parquet

report   = validate_nyc_taxi_data(df_row_train)
passed_n = sum(r['passed'] for r in report['results'])
total_n  = len(report['results'])
status   = 'ALL PASSED' if report['success'] else 'SOME CHECKS FAILED'
print(f'Result: {status}  ({passed_n}/{total_n} checks passed)')

Result: SOME CHECKS FAILED  (10/12 checks passed)


In [8]:
# Full results table
print('{:<4} {:<32} {:<48} {}'.format('#', 'Column', 'Check', 'Result'))
print('-' * 95)
for i, r in enumerate(report['results'], 1):
    col    = r['column']
    name   = r['name']
    icon   = 'OK  ' if r['passed'] else 'FAIL'
    print('{:<4} {:<32} {:<48} {}'.format(i, col, name, icon))
    if not r['passed']:
        detail = r['detail']
        print(f'     >> {detail}')

#    Column                           Check                                            Result
-----------------------------------------------------------------------------------------------
1    tpep_pickup_datetime             not_null                                         OK  
2    tpep_dropoff_datetime            not_null                                         OK  
3    tpep_pickup_datetime, tpep_dropoff_datetime tpep_dropoff_datetime_geq_tpep_pickup_datetime   FAIL
     >> 0.00% of rows violate ordering constraint (tpep_dropoff_datetime < tpep_pickup_datetime)
4    PULocationID                     not_null                                         OK  
5    PULocationID                     between[1,265]                                   OK  
6    DOLocationID                     not_null                                         OK  
7    DOLocationID                     between[1,265]                                   OK  
8    trip_distance                    between[0.0,None]   

<a id="section-1-4"></a>
## 1.4 Data Cleaning

In [9]:
print("--- Cleaning TRAIN Data ---")
df_clean_train = clean_parquet(DF_not_clean=df_row_train,output_path=TRAIN_CLEANED_PARQUET, is_train=True)

print("\n--- Cleaning TEST Data ---")
df_clean_test  = clean_parquet(DF_not_clean=df_row_test, output_path=TEST_CLEANED_PARQUET, is_train=False)

print("\n" + "="*40)
print(f"Train shape after cleaning: {df_clean_train.shape}")
print(f"Test shape after cleaning:  {df_clean_test.shape}")
print("="*40)

--- Cleaning TRAIN Data ---

STARTING CLEANING PIPELINE
Loading data from DF...
Processing DataFrame...
  Input  rows : 41,169,720
  drop_critical_nulls  : no rows dropped
  drop_pre_december_2023  : dropped 39 rows
  fill_non_critical_nulls : filled - RatecodeID (4,091,232 -> 1), store_and_fwd_flag (4,091,232 -> 'N'), Airport_fee (4,091,232 -> 0.0)
  select_relevant_columns : dropped 9 columns - ['VendorID', 'passenger_count', 'store_and_fwd_flag', 'extra', 'mta_tax', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'Airport_fee']
  drop_remaining_nulls : no remaining nulls
  filter_temporal_logic   : dropped 1,575 invalid rows
  filter_monetary_logic   : dropped 851,905 invalid rows
  Output rows : 40,316,201
  Saved cleaned file -> data/processed/DF_test_2024_2025_cleaned.parquet


--- Cleaning TEST Data ---

STARTING CLEANING PIPELINE
Loading data from DF...
Processing DataFrame...
  Input  rows : 14,177,524
  fill_non_critical_nulls : filled - RatecodeID (3,458,461

In [15]:
df_clean_train.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,PULocationID,DOLocationID,RatecodeID,payment_type,fare_amount,total_amount,tip_amount
0,2024-01-01 00:57:55,2024-01-01 01:17:43,1.72,186,79,1.0,2,17.7,22.70,0.00
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.80,140,236,1.0,1,10.0,18.75,3.75
2,2024-01-01 00:17:06,2024-01-01 00:35:01,4.70,236,79,1.0,1,23.3,31.30,3.00
3,2024-01-01 00:36:38,2024-01-01 00:44:56,1.40,79,211,1.0,1,10.0,17.00,2.00
4,2024-01-01 00:46:51,2024-01-01 00:52:57,0.80,211,148,1.0,1,7.9,16.10,3.20


<a id="section-1-5"></a>

## 1.5  Feature engineering

In [16]:
# ══════════════════════════════════════════════════════════════════════════
# EXPERIMENT B — Full Feature Engineering
# ══════════════════════════════════════════════════════════════════════════

_print_header("STEP 4 — Experiment B: Full Feature Engineering")

eng_train, eng_scaler = run_feature_pipeline(df_clean_train, is_training=True, scaler_save_path=SCALER_SAVE_PATH_ENGINEERD)
X_train_eng = eng_train.drop(columns=[TARGET_COL])
y_train_eng = eng_train[TARGET_COL]
print(f"  Engineered feature columns ({len(X_train_eng.columns)}): " f"{X_train_eng.columns.tolist()}")


eng_test, _ = run_feature_pipeline(df_clean_test, scaler=eng_scaler, is_training=False)
X_test_eng  = eng_test.drop(columns=[TARGET_COL])
y_test_eng  = eng_test[TARGET_COL]


# Save the fitted scaler so the Streamlit app can load it without rerunning the pipeline
Path(MODEL_DIR_ENGINEERED).mkdir(parents=True, exist_ok=True)
joblib.dump(eng_scaler, Path(MODEL_DIR_ENGINEERED) / "scaler.pkl")
print(f"  Scaler saved → {MODEL_DIR_ENGINEERED}/scaler.pkl")


# ══════════════════════════════════════════════════════════════════════════
print("\n" + "~"*18)
print("X_train_base sample:")
print("~"*18)

display(X_train_eng.head(500))

print("\n" + "-"*25)

print("\n" + "~"*18)
print("y_train_base sample:")
print("~"*18)
display(y_train_eng.head(6))

# display(X_train_eng['pickup_hour'].value_counts().sort_index()) # to check if the values are in the right order (0-23) and if the distribution makes sense
# display(X_train_eng['day_of_week'].value_counts().sort_index()) # to check if the values are in the right order (0-6) and if the distribution makes sense
# display(X_train_eng['is_weekend'].value_counts().sort_index()) # to check if the values are in the right order (0-9) and if the distribution makes sense
# display(X_train_eng['is_rush_hour'].value_counts().sort_index()) # to check if the values are in the right order (0-4) and if the distribution makes sense
# display(X_train_eng['time_of_day_bucket'].value_counts().sort_index()) # to check if the values are in the right order and if the distribution makes sense


STEP 4 — Experiment B: Full Feature Engineering
  Saved feature scaler -> data/feature_stores/engineered_scaler.pkl
  Engineered feature columns (17): ['trip_distance', 'PULocationID', 'DOLocationID', 'pickup_hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'time_of_day_bucket', 'est_base_fare', 'est_congestion_surcharge', 'est_extra', 'est_airport_fee', 'est_improvement_surcharge', 'est_mta_tax', 'est_total_fare_without_tolls', 'pickup_hour_sin', 'pickup_hour_cos']
  Scaler saved → data/models/engineered_model/scaler.pkl

~~~~~~~~~~~~~~~~~~
X_train_base sample:
~~~~~~~~~~~~~~~~~~


,trip_distance,PULocationID,DOLocationID,pickup_hour,day_of_week,is_weekend,is_rush_hour,time_of_day_bucket,est_base_fare,est_congestion_surcharge,est_extra,est_airport_fee,est_improvement_surcharge,est_mta_tax,est_total_fare_without_tolls,pickup_hour_sin,pickup_hour_cos
0,-0.008708,186,79,-2.457426,-1.598947,-0.629142,0,-2.135614,-0.008853,0.255585,0.203823,-0.250832,1.0,0.5,-0.008940,0.467437,1.678554
1,-0.008406,140,236,-2.457426,-1.598947,-0.629142,0,-2.135614,-0.008546,0.255585,0.203823,-0.250832,1.0,0.5,-0.008633,0.467437,1.678554
2,0.002538,236,79,-2.457426,-1.598947,-0.629142,0,-2.135614,0.002582,0.255585,0.203823,-0.250832,1.0,0.5,0.002496,0.467437,1.678554
3,-0.009916,79,211,-2.457426,-1.598947,-0.629142,0,-2.135614,-0.010081,0.255585,0.203823,-0.250832,1.0,0.5,-0.010168,0.467437,1.678554
4,-0.012180,211,148,-2.457426,-1.598947,-0.629142,0,-2.135614,-0.012384,0.255585,0.203823,-0.250832,1.0,0.5,-0.012470,0.467437,1.678554
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,-0.001236,148,237,-2.457426,-1.598947,-0.629142,0,-2.135614,-0.001255,0.255585,0.203823,-0.250832,1.0,0.5,-0.001342,0.467437,1.678554
496,-0.004670,142,166,-2.457426,-1.598947,-0.629142,0,-2.135614,-0.004747,0.255585,0.203823,-0.250832,1.0,0.5,-0.004834,0.467437,1.678554
497,-0.011803,166,238,-2.457426,-1.598947,-0.629142,0,-2.135614,-0.012000,0.255585,0.203823,-0.250832,1.0,0.5,-0.012086,0.467437,1.678554
498,-0.011727,238,75,-2.457426,-1.598947,-0.629142,0,-2.135614,-0.011923,0.255585,0.203823,-0.250832,1.0,0.5,-0.012010,0.467437,1.678554



-------------------------

~~~~~~~~~~~~~~~~~~
y_train_base sample:
~~~~~~~~~~~~~~~~~~


,total_fare_amount
0,22.7
1,15.0
2,28.3
3,15.0
4,12.9
5,34.6


# EDA research

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import glob
import pyarrow.parquet as pq

# Set style for plots
sns.set_style("whitegrid")

# Create an unscaled dataset for plotting to ensure variables have intuitive ranges/meanings
eda_plot_df = df_clean_train.copy()
if "total_fare_amount" not in eda_plot_df.columns:
    eda_plot_df["total_fare_amount"] = eda_plot_df["total_amount"] - eda_plot_df["tip_amount"]
if "pickup_hour" not in eda_plot_df.columns:
    eda_plot_df["pickup_hour"] = eda_plot_df["tpep_pickup_datetime"].dt.hour
if "day_of_week" not in eda_plot_df.columns:
    eda_plot_df["day_of_week"] = eda_plot_df["tpep_pickup_datetime"].dt.dayofweek
if "is_weekend" not in eda_plot_df.columns:
    eda_plot_df["is_weekend"] = (eda_plot_df["day_of_week"] >= 5).astype(int)

# Plot 1: Distribution of total_fare_amount (target variable)
plt.figure(figsize=(10, 6))
sns.histplot(eda_plot_df['total_fare_amount'], bins=50, kde=True, color='purple')
plt.title('Distribution of Total Fare Amount (Train Data)')
plt.xlabel('Total Fare Amount ($)')
plt.ylabel('Count')
plt.xlim(0, eda_plot_df['total_fare_amount'].quantile(0.995))
plt.show()

# Plot 2: Distribution of original trip_distance
plt.figure(figsize=(10, 6))
sns.histplot(eda_plot_df['trip_distance'], bins=50, kde=True, color='green')
plt.title('Distribution of Trip Distance (Cleaned Train Data)')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Count')
plt.xlim(0, eda_plot_df['trip_distance'].quantile(0.995))
plt.show()

# Plot 3: Average total_fare_amount by pickup_hour
plt.figure(figsize=(12, 6))
sns.barplot(x='pickup_hour', y='total_fare_amount', data=eda_plot_df.sort_values('pickup_hour'), errorbar=None, color='skyblue')
plt.title('Average Total Fare Amount by Pickup Hour')
plt.xlabel('Pickup Hour (0-23)')
plt.ylabel('Average Total Fare Amount ($)')
plt.show()

# Plot 4: Average total_fare_amount by day_of_week
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
eda_plot_df['day_of_week_name'] = eda_plot_df['day_of_week'].map(lambda x: day_names[x])

plt.figure(figsize=(12, 6))
sns.barplot(x='day_of_week_name', y='total_fare_amount', data=eda_plot_df.sort_values('day_of_week'), errorbar=None, color='lightcoral')
plt.title('Average Total Fare Amount by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Total Fare Amount ($)')
plt.show()

# Plot 5: Count of is_weekend
plt.figure(figsize=(8, 5))
sns.countplot(x='is_weekend', data=eda_plot_df, hue='is_weekend', palette='viridis', legend=False)
plt.title('Count of Rides: Weekday vs. Weekend')
plt.xlabel('Is Weekend (0=Weekday, 1=Weekend)')
plt.ylabel('Number of Rides')
plt.xticks([0, 1], ['Weekday', 'Weekend'])
plt.show()

# Plot 6: Average total_fare_amount by payment_type
plt.figure(figsize=(10, 6))
sns.barplot(x='payment_type', y='total_fare_amount', data=eda_plot_df.sort_values('payment_type'), errorbar=None, palette='pastel')
plt.title('Average Total Fare Amount by Payment Type')
plt.xlabel('Payment Type')
plt.ylabel('Average Total Fare Amount ($)')
plt.show()

# Plot 7: Data Size Summary Before and After Cleaning
train_files = glob.glob("data/raw/train/*.parquet")
test_files = glob.glob("data/raw/test/*.parquet")

train_raw_rows = sum(pq.read_metadata(f).num_rows for f in train_files)
test_raw_rows = sum(pq.read_metadata(f).num_rows for f in test_files)

quality_summary = pd.DataFrame({
    "dataset": ["Raw Train Data", "Cleaned Train Data", "Raw Test Data", "Cleaned Test Data"],
    "rows": [
        train_raw_rows,
        len(df_clean_train),
        test_raw_rows,
        len(df_clean_test)
    ]
})

display(quality_summary)

plt.figure(figsize=(10, 5))
sns.barplot(data=quality_summary, x="dataset", y="rows", color="steelblue")
plt.title("Plot 7: Data Size Summary Before and After Cleaning")
plt.xlabel("Dataset Version")
plt.ylabel("Number of Rows")
plt.ticklabel_format(style="plain", axis="y")
plt.xticks(rotation=20)
plt.show()

# Plot 8: Trip Distance vs Mean Fare by Distance Bucket
df_distance_bucket = eda_plot_df.copy()
df_distance_bucket = df_distance_bucket[
    (df_distance_bucket["trip_distance"] > 0) &
    (df_distance_bucket["trip_distance"] <= 50)
]
df_distance_bucket["distance_bucket"] = pd.cut(
    df_distance_bucket["trip_distance"],
    bins=range(0, 51, 1),
    right=False
)

distance_fare_summary = (
    df_distance_bucket
    .groupby("distance_bucket", observed=True)
    .agg(
        trips=("total_fare_amount", "count"),
        mean_fare=("total_fare_amount", "mean"),
        median_fare=("total_fare_amount", "median")
    )
    .reset_index()
)

distance_fare_summary["distance_bucket_mid"] = distance_fare_summary["distance_bucket"].apply(
    lambda x: x.left + 0.5
)

plt.figure(figsize=(12, 5))
plt.plot(
    distance_fare_summary["distance_bucket_mid"],
    distance_fare_summary["mean_fare"],
    marker="o",
    linewidth=2,
    label="Mean fare"
)
plt.plot(
    distance_fare_summary["distance_bucket_mid"],
    distance_fare_summary["median_fare"],
    marker="o",
    linewidth=2,
    label="Median fare"
)
plt.title("Plot 8: Mean and Median Fare by Trip Distance Bucket")
plt.xlabel("Trip Distance Bucket Midpoint (miles)")
plt.ylabel("Total Fare Amount ($)")
plt.legend()
plt.grid(True)
plt.show()

display(distance_fare_summary.head(10))

# Plot 9: Overnight Surcharge Analysis
eda_plot_df["is_overnight"] = (
    (eda_plot_df["pickup_hour"].astype(int) >= 20) |
    (eda_plot_df["pickup_hour"].astype(int) < 6)
)

overnight_summary = (
    eda_plot_df
    .groupby("is_overnight")
    .agg(
        trips=("total_fare_amount", "count"),
        mean_fare=("total_fare_amount", "mean"),
        median_fare=("total_fare_amount", "median"),
        mean_distance=("trip_distance", "mean")
    )
    .reset_index()
)

overnight_summary["ride_time"] = overnight_summary["is_overnight"].map({
    False: "Non-overnight ride",
    True: "Overnight ride"
})

plt.figure(figsize=(9, 5))
sns.barplot(data=overnight_summary, x="ride_time", y="mean_fare", color="darkorange")
plt.title("Plot 9: Mean Fare for Overnight vs Non-Overnight Rides")
plt.xlabel("Ride Time")
plt.ylabel("Mean Total Fare Amount ($)")
plt.show()

display(overnight_summary)

# Plot 10: Rush Hour Analysis
if "day_of_week_num" not in eda_plot_df.columns:
    eda_plot_df["day_of_week_num"] = eda_plot_df["day_of_week"]

eda_plot_df["is_rush_hour"] = (
    (eda_plot_df["day_of_week_num"].astype(int).between(0, 4)) &
    (eda_plot_df["pickup_hour"].astype(int) >= 16) &
    (eda_plot_df["pickup_hour"].astype(int) < 20)
)

rush_hour_summary = (
    eda_plot_df
    .groupby("is_rush_hour")
    .agg(
        trips=("total_fare_amount", "count"),
        mean_fare=("total_fare_amount", "mean"),
        median_fare=("total_fare_amount", "median"),
        mean_distance=("trip_distance", "mean")
    )
    .reset_index()
)

rush_hour_summary["ride_period"] = rush_hour_summary["is_rush_hour"].map({
    False: "Non-rush-hour ride",
    True: "Rush-hour ride"
})

plt.figure(figsize=(9, 5))
sns.barplot(data=rush_hour_summary, x="ride_period", y="mean_fare", color="firebrick")
plt.title("Plot 10: Mean Fare for Rush Hour vs Non-Rush Hour Rides")
plt.xlabel("Ride Period")
plt.ylabel("Mean Total Fare Amount ($)")
plt.show()

display(rush_hour_summary)

# Plot 11: Holiday vs Non-Holiday Analysis
holiday_dates = pd.to_datetime([
    "2024-01-01", "2024-01-15", "2024-02-19", "2024-05-27", "2024-07-04",
    "2024-09-02", "2024-10-14", "2024-11-11", "2024-11-28", "2024-12-25",
    "2025-01-01", "2025-01-20", "2025-02-17", "2025-05-26", "2025-07-04",
    "2025-09-01", "2025-10-13", "2025-11-11", "2025-11-27", "2025-12-25"
])

eda_plot_df["pickup_date"] = eda_plot_df["tpep_pickup_datetime"].dt.normalize()
eda_plot_df["is_holiday"] = eda_plot_df["pickup_date"].isin(holiday_dates)

holiday_summary = (
    eda_plot_df
    .groupby("is_holiday")
    .agg(
        trips=("total_fare_amount", "count"),
        mean_fare=("total_fare_amount", "mean"),
        median_fare=("total_fare_amount", "median"),
        mean_distance=("trip_distance", "mean")
    )
    .reset_index()
)

holiday_summary["day_type"] = holiday_summary["is_holiday"].map({
    False: "Non-holiday",
    True: "Holiday"
})

plt.figure(figsize=(9, 5))
sns.barplot(data=holiday_summary, x="day_type", y="mean_fare", color="mediumpurple")
plt.title("Plot 11: Mean Fare for Holiday vs Non-Holiday Rides")
plt.xlabel("Day Type")
plt.ylabel("Mean Total Fare Amount ($)")
plt.show()

display(holiday_summary)

# Plot 12: Airport Rides Analysis
airport_zones = {132, 138, 1}
eda_plot_df["is_airport"] = (
    eda_plot_df["PULocationID"].isin(airport_zones) |
    eda_plot_df["DOLocationID"].isin(airport_zones)
)

airport_summary = (
    eda_plot_df
    .groupby("is_airport")
    .agg(
        trips=("total_fare_amount", "count"),
        mean_fare=("total_fare_amount", "mean"),
        median_fare=("total_fare_amount", "median"),
        mean_distance=("trip_distance", "mean")
    )
    .reset_index()
)

airport_summary["ride_type"] = airport_summary["is_airport"].map({
    False: "Non-airport ride",
    True: "Airport ride"
})

plt.figure(figsize=(9, 5))
sns.barplot(data=airport_summary, x="ride_type", y="mean_fare", color="steelblue")
plt.title("Plot 12: Mean Fare for Airport vs Non-Airport Rides")
plt.xlabel("Ride Type")
plt.ylabel("Mean Total Fare Amount ($)")
plt.show()

display(airport_summary)
